# ConvNeXt + MaxSig — Stage-Placement Ablation

`ConvNext-MaxSig.ipynb` injects a single `MSLNAdapter` (a bottleneck-reduced MaxSig adapter:
1x1 reduce -> `MaxSigLayerNormalized` -> 1x1 expand, gated by a learnable `alpha` starting at 0)
into one ConvNeXt-tiny backbone stage. The other three stages are left as `nn.Identity()`. The
**default** configuration applies the adapter at **stage 2 only (384 channels)** —
`kernel_size=5`, `use_sigmoid_input=False`, `bottleneck_ratio=4`.

| stage index | channels | relative resolution |
|---|---|---|
| 0 | 96  | highest (stem/4)  |
| 1 | 192 | H/8  |
| 2 | 384 | H/16 (**default MaxSig location**) |
| 3 | 768 | lowest (H/32) |

This notebook keeps that default and adds the **two additional scenarios a reviewer is most
likely to ask for** — bracketing the chosen stage from both ends, the same logic used for the
Swin-UNet ablation:

1. **Default (unchanged)** — MaxSig adapter at Stage 2 only (384ch)
2. **MaxSig at Stage 0 only (96ch, highest resolution)** — reviewer framing: *"Did you test
   applying MaxSig at the earliest, highest-resolution stage and find it hurt performance?"* —
   the ConvNeXt analogue of the Stage-0 question raised for the plain U-Net.
3. **MaxSig at Stage 3 only (768ch, deepest stage)** — reviewer framing: *"Why stage 2 and not the
   deepest stage? Did you test placing the adapter at the very end of the backbone instead?"*

Each scenario is its own cell and reports only: **best validation Dice**, **best epoch**, and
**test Dice**. For the **default configuration** it additionally reports **GFLOPs** and **FPS**.


In [1]:
# ============================================================
# Config
# ============================================================
ROOT_DIR_STR = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/compsitdataset/dataset'

config = {
    "lr": 1e-4,
    "batch_size": 8,
    "epochs": 100,          # lower this (e.g. 40-60) for a faster 3-run sweep if needed
    "img_size": 512,
    "backbone": "tiny",
}

RESULTS_DIR_STR = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/compsitdataset/convnext_ablation_ckpts/'


In [2]:
import os
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

ROOT_DIR = Path(ROOT_DIR_STR)
RESULTS_DIR = Path(RESULTS_DIR_STR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", device)


device: cuda


## Metrics\n(unchanged from `ConvNext-MaxSig.ipynb`)

In [3]:
@torch.no_grad()
def dice_from_logits(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dims)
    denom = preds.sum(dims) + targets.sum(dims)
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean()


@torch.no_grad()
def iou_from_logits(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dims)
    union = preds.sum(dims) + targets.sum(dims) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, device: str = "cuda") -> dict:
    model.eval()
    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    n = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device).float().unsqueeze(1)

        logits = model(images)
        logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

        loss = criterion(logits, masks).item()
        d = dice_from_logits(logits, masks).item()
        j = iou_from_logits(logits, masks).item()

        total_loss += loss
        total_dice += d
        total_iou += j
        n += 1

    if n == 0:
        return {"loss": 0.0, "dice": 0.0, "iou": 0.0}

    return {"loss": total_loss / n, "dice": total_dice / n, "iou": total_iou / n}


## MaxSig components\n(unchanged from `ConvNext-MaxSig.ipynb`)

In [4]:
class MaxSigLayerNormalized(nn.Module):
    def __init__(self, ch_in: int, ch_out: int, kernel_size: int, stride: int = 1, use_sigmoid_input: bool = True):
        super().__init__()
        assert kernel_size >= 1
        self.ch_in = ch_in
        self.ch_out = ch_out
        self.kernel_size = kernel_size
        self.stride = stride
        self.use_sigmoid_input = use_sigmoid_input

        self.padding = (kernel_size - 1) // 2
        self.kernel_area = kernel_size ** 2
        self.center_index = self.kernel_area // 2

        self.weight = nn.Parameter(torch.rand(1, 1, self.kernel_area, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, 1, self.kernel_area, 1, 1))

        self.weight_center = nn.Parameter(torch.tensor(1.0))
        self.weight_median = nn.Parameter(torch.tensor(1.0))

        self.proj = nn.Conv2d(ch_in, ch_out, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        assert C_in == self.ch_in, f"Expected {self.ch_in} channels, got {C_in}"

        x_in = torch.sigmoid(x) if self.use_sigmoid_input else x

        x_unfold = F.unfold(x_in, kernel_size=self.kernel_size, padding=self.padding, stride=self.stride)
        _, CK2, L = x_unfold.shape
        assert CK2 == C_in * self.kernel_area

        out_h = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (W + 2 * self.padding - self.kernel_size) // self.stride + 1

        x_unfold = x_unfold.view(B, C_in, self.kernel_area, out_h, out_w)

        w = torch.sigmoid(self.weight)
        b = torch.sigmoid(self.bias)

        D = torch.maximum(w, x_unfold) + b

        mean_stat = torch.mean(D, dim=2)
        median_stat, _ = torch.median(D, dim=2)

        center_values = x_unfold[:, :, self.center_index, :, :]

        out = (self.weight_center * center_values + self.weight_median * median_stat - mean_stat)
        out = self.proj(out)
        return out


class MSLNAdapter(nn.Module):
    """Bottleneck adapter: reduce -> MaxSig -> expand, gated by alpha (starts at 0 => identity
    at init, very stable)."""
    def __init__(self, dim, bottleneck_ratio=4, kernel_size=5, use_sigmoid_input=False):
        super().__init__()
        b = max(8, dim // bottleneck_ratio)
        self.reduce = nn.Conv2d(dim, b, kernel_size=1, bias=False)
        self.msln   = MaxSigLayerNormalized(b, b, kernel_size=kernel_size, stride=1, use_sigmoid_input=use_sigmoid_input)
        self.expand = nn.Conv2d(b, dim, kernel_size=1, bias=False)
        self.alpha  = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        return x + self.alpha * self.expand(self.msln(self.reduce(x)))


## ConvNeXt backbone, UPerNet decoder, full model\n(unchanged from `ConvNext-MaxSig.ipynb`, **except** `ConvNeXtBackbone` now takes `maxsig_stages: tuple` instead of a hardcoded per-stage adapter list, so any subset of stages can be enabled — `(2,)` exactly reproduces the original default.)

In [3]:
class LayerNorm(nn.Module):
    """ConvNeXt-style LayerNorm supporting channels_last (N,H,W,C) and channels_first (N,C,H,W)."""
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        if data_format not in ("channels_last", "channels_first"):
            raise ValueError("data_format must be 'channels_last' or 'channels_first'")
        self.data_format = data_format
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == "channels_last":
            mean = x.mean(-1, keepdim=True)
            var = (x - mean).pow(2).mean(-1, keepdim=True)
            x = (x - mean) / torch.sqrt(var + self.eps)
            x = x * self.weight + self.bias
            return x
        else:
            mean = x.mean(1, keepdim=True)
            var = (x - mean).pow(2).mean(1, keepdim=True)
            x = (x - mean) / torch.sqrt(var + self.eps)
            x = x * self.weight[:, None, None] + self.bias[:, None, None]
            return x


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path=0., layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm(dim, eps=1e-6, data_format="channels_last")
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones(dim), requires_grad=True) \
            if layer_scale_init_value > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        x = residual + self.drop_path(x)
        return x


class ConvNeXtBackbone(nn.Module):
    """maxsig_stages: tuple of stage indices (subset of {0,1,2,3}) to wrap with an
    MSLNAdapter. () = plain ConvNeXt, no MaxSig anywhere. (2,) reproduces the original default."""
    def __init__(self, in_chans=1, depths=[3, 3, 9, 3], dims=[96, 192, 384, 768],
                 drop_path_rate=0., layer_scale_init_value=1e-6,
                 maxsig_stages=(2,), maxsig_kernel_size=5, maxsig_use_sigmoid_input=False,
                 maxsig_bottleneck_ratio=4):
        super().__init__()

        self.downsample_layers = nn.ModuleList()
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            LayerNorm(dims[0], eps=1e-6, data_format="channels_first"),
        )
        self.downsample_layers.append(stem)

        for i in range(1, 4):
            down = nn.Sequential(
                LayerNorm(dims[i - 1], eps=1e-6, data_format="channels_first"),
                nn.Conv2d(dims[i - 1], dims[i], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(down)

        self.stages = nn.ModuleList()
        dp_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i in range(4):
            blocks = nn.Sequential(*[
                ConvNeXtBlock(
                    dim=dims[i],
                    drop_path=dp_rates[cur + j],
                    layer_scale_init_value=layer_scale_init_value
                )
                for j in range(depths[i])
            ])
            self.stages.append(blocks)
            cur += depths[i]

        self.out_norms = nn.ModuleList([LayerNorm(dims[i], eps=1e-6, data_format="channels_first") for i in range(4)])

        maxsig_stages = set(maxsig_stages)
        self.adapters = nn.ModuleList([
            MSLNAdapter(dims[i], bottleneck_ratio=maxsig_bottleneck_ratio,
                        kernel_size=maxsig_kernel_size, use_sigmoid_input=maxsig_use_sigmoid_input)
            if i in maxsig_stages else nn.Identity()
            for i in range(4)
        ])

    def forward(self, x):
        outs = []
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
            x = self.adapters[i](x)
            outs.append(self.out_norms[i](x))
        return outs


class UPerNetDecoder(nn.Module):
    def __init__(self, in_channels=[96, 192, 384, 768], channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([nn.Conv2d(in_ch, channels, 1) for in_ch in in_channels])
        self.fpn_convs = nn.ModuleList([nn.Conv2d(channels, channels, 3, padding=1) for _ in in_channels])

        self.psp_pooling = nn.AdaptiveAvgPool2d(1)
        self.psp_conv = nn.Conv2d(in_channels[-1], channels, 1)

        self.fusion = nn.Conv2d(channels, channels, 3, padding=1)
        self.cls = nn.Conv2d(channels, 1, 1)

    def forward(self, features):
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        for i in range(len(laterals) - 1, 0, -1):
            laterals[i - 1] = laterals[i - 1] + F.interpolate(
                laterals[i], size=laterals[i - 1].shape[2:], mode="bilinear", align_corners=False
            )

        fpn_outs = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]

        target_size = fpn_outs[0].shape[2:]
        fpn_outs = [
            o if o.shape[2:] == target_size else F.interpolate(o, size=target_size, mode="bilinear", align_corners=False)
            for o in fpn_outs
        ]

        psp = self.psp_conv(self.psp_pooling(features[-1]))
        psp = F.interpolate(psp, size=target_size, mode="bilinear", align_corners=False)

        fused = self.fusion(sum(fpn_outs) / len(fpn_outs) + psp)
        return self.cls(fused)


class ConvNeXtBinarySegmentation(nn.Module):
    def __init__(self, backbone="tiny", maxsig_stages=(2,), maxsig_kernel_size=5,
                 maxsig_use_sigmoid_input=False, maxsig_bottleneck_ratio=4):
        super().__init__()
        if backbone == "tiny":
            depths, dims = [3, 3, 9, 3], [96, 192, 384, 768]
        elif backbone == "small":
            depths, dims = [3, 3, 27, 3], [96, 192, 384, 768]
        elif backbone == "base":
            depths, dims = [3, 3, 27, 3], [128, 256, 512, 1024]
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

        self.backbone = ConvNeXtBackbone(
            depths=depths, dims=dims,
            maxsig_stages=maxsig_stages, maxsig_kernel_size=maxsig_kernel_size,
            maxsig_use_sigmoid_input=maxsig_use_sigmoid_input,
            maxsig_bottleneck_ratio=maxsig_bottleneck_ratio,
        )
        self.decoder = UPerNetDecoder(in_channels=dims)

    def forward(self, x):
        features = self.backbone(x)
        return self.decoder(features)


## Dataset\n(unchanged from `ConvNext-MaxSig.ipynb`)

In [6]:
class BinarySegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=512, augment=False):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.augment = augment
        self.mask_names = sorted(list(self.mask_dir.glob('*.tif')) + list(self.mask_dir.glob('*.bmp')) + list(self.mask_dir.glob('*.png')))

    def __len__(self):
        return len(self.mask_names)

    def __getitem__(self, index):
        mask_name = self.mask_names[index]
        mask = cv2.imread(mask_name, cv2.IMREAD_ANYDEPTH)
        mask = (mask - mask.min()) / (mask.max() - mask.min())
        mask = (mask > 0).astype(np.float32)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_AREA)
        mask = torch.from_numpy(mask)

        imgn = str(mask_name).replace('/masks/', '/images/')
        if imgn.endswith('.tif'):
            if not os.path.exists(imgn):
                imgn = imgn.replace('.tif', '.bmp')

        image = cv2.imread(imgn, cv2.IMREAD_ANYDEPTH)
        image = (image - image.min()) / (image.max() - image.min()).astype(np.float32)
        image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_NEAREST)
        image = np.expand_dims(image, axis=0)
        image = torch.from_numpy(image)

        return image, mask


train_dataset_ds = BinarySegmentationDataset(ROOT_DIR / "train" / "images", ROOT_DIR / "train" / "masks", img_size=config["img_size"])
val_dataset_ds   = BinarySegmentationDataset(ROOT_DIR / "val"   / "images", ROOT_DIR / "val"   / "masks", img_size=config["img_size"])
test_dataset_ds  = BinarySegmentationDataset(ROOT_DIR / "test"  / "images", ROOT_DIR / "test"  / "masks", img_size=config["img_size"])

train_loader = DataLoader(train_dataset_ds, batch_size=config["batch_size"], shuffle=True,  num_workers=0, pin_memory=(device == "cuda"))
val_loader   = DataLoader(val_dataset_ds,   batch_size=config["batch_size"], shuffle=False, num_workers=0, pin_memory=(device == "cuda"))
test_loader  = DataLoader(test_dataset_ds,  batch_size=config["batch_size"], shuffle=False, num_workers=0, pin_memory=(device == "cuda"))

print("train:", len(train_dataset_ds), " val:", len(val_dataset_ds), " test:", len(test_dataset_ds))


train: 294  val: 63  test: 63


## Ablation helpers

`run_scenario(run_name, maxsig_stages, ...)` builds a fresh `ConvNeXtBinarySegmentation` with the
`MSLNAdapter` at the given backbone stage(s) (`()` = plain ConvNeXt, no MaxSig), trains it,
evaluates test Dice with the best checkpoint, and reports only `best_val_dice`, `best_epoch`,
`test_dice`.

`compute_flops_fps(...)` reports GFLOPs (via `thop`) and FPS for a model — used only for the
default configuration.


In [6]:
def run_scenario(run_name, maxsig_stages, epochs=config["epochs"], verbose=True):
    torch.manual_seed(42)

    model = ConvNeXtBinarySegmentation(
        backbone=config["backbone"], maxsig_stages=maxsig_stages,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": config["lr"]},
            {"params": model.decoder.parameters(), "lr": config["lr"]},
        ],
        weight_decay=0.05,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    ckpt_path = RESULTS_DIR / f"best-{run_name}.pth"
    best_dice = -1.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            logits = model(images)
            logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()
        scheduler.step()

        val_stats = evaluate_model(model, val_loader, device=device)
        val_dice = val_stats["dice"]

        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch + 1
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_dice": val_dice}, ckpt_path)

        if verbose and (epoch + 1) % 10 == 0:
            print(f"[{run_name}] epoch {epoch+1}/{epochs} | val_dice={val_dice:.4f} | best={best_dice:.4f}@{best_epoch}")

    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_stats = evaluate_model(model, test_loader, device=device)
    test_dice = test_stats["dice"]

    result = {
        "run_name": run_name,
        "maxsig_stages": tuple(sorted(maxsig_stages)),
        "best_val_dice": best_dice,
        "best_epoch": best_epoch,
        "test_dice": test_dice,
    }
    print(f"[{run_name}] DONE | best_val_dice={best_dice:.4f} (epoch {best_epoch}) | test_dice={test_dice:.4f}")
    return result, model


def compute_flops_fps(model, input_size=(1, 1, 512, 512), n_warmup=10, n_iters=50):
    """Reports GFLOPs (thop) and FPS (measured wall-clock throughput) for a model."""
    model.eval()
    dummy = torch.randn(*input_size).to(device)

    try:
        from thop import profile
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        gflops = macs * 2 / 1e9
    except ImportError:
        print("`thop` not installed. Run `pip install thop` to get GFLOPs. Skipping GFLOPs.")
        gflops = None

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = perf_counter()
        for _ in range(n_iters):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = perf_counter()

    fps = n_iters / (t1 - t0)
    return gflops, fps


ablation_results = []


### Scenario 1 — Default: MaxSig adapter at Stage 2 only (384ch)\n(the configuration already used in `ConvNext-MaxSig.ipynb`)

In [8]:
res, default_model = run_scenario(run_name="default_stage2_only_384ch", maxsig_stages=(2,))
ablation_results.append(res)


/tmp/ipykernel_434032/3248516051.py:15: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[default_stage2_only_384ch] epoch 10/100 | val_dice=0.5064 | best=0.5260@7
[default_stage2_only_384ch] epoch 20/100 | val_dice=0.5559 | best=0.5636@15
[default_stage2_only_384ch] epoch 30/100 | val_dice=0.5531 | best=0.5707@23
[default_stage2_only_384ch] epoch 40/100 | val_dice=0.5171 | best=0.5707@23
[default_stage2_only_384ch] epoch 50/100 | val_dice=0.5313 | best=0.5707@23
[default_stage2_only_384ch] epoch 60/100 | val_dice=0.5220 | best=0.5707@23
[default_stage2_only_384ch] epoch 70/100 | val_dice=0.5138 | best=0.5707@23
[default_stage2_only_384ch] epoch 80/100 | val_dice=0.5069 | best=0.5707@23
[default_stage2_only_384ch] epoch 90/100 | val_dice=0.5048 | best=0.5707@23
[default_stage2_only_384ch] epoch 100/100 | val_dice=0.5044 | best=0.5707@23
[default_stage2_only_384ch] DONE | best_val_dice=0.5707 (epoch 23) | test_dice=0.5387


### Scenario 2 — MaxSig at **Stage 0 only (96ch, highest resolution)**
> Reviewer framing: *"Did you test applying MaxSig at the earliest, highest-resolution stage and
> find it hurt performance?"* — the ConvNeXt analogue of the Stage-0 question raised for the
> plain U-Net.


In [9]:
res, _ = run_scenario(run_name="stage0_only_96ch", maxsig_stages=(0,))
ablation_results.append(res)


/tmp/ipykernel_434032/3248516051.py:15: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage0_only_96ch] epoch 10/100 | val_dice=0.5325 | best=0.5325@10
[stage0_only_96ch] epoch 20/100 | val_dice=0.5615 | best=0.5775@18
[stage0_only_96ch] epoch 30/100 | val_dice=0.5473 | best=0.5795@26
[stage0_only_96ch] epoch 40/100 | val_dice=0.5577 | best=0.5795@26
[stage0_only_96ch] epoch 50/100 | val_dice=0.5433 | best=0.5795@26
[stage0_only_96ch] epoch 60/100 | val_dice=0.5322 | best=0.5795@26
[stage0_only_96ch] epoch 70/100 | val_dice=0.5269 | best=0.5795@26
[stage0_only_96ch] epoch 80/100 | val_dice=0.5387 | best=0.5795@26
[stage0_only_96ch] epoch 90/100 | val_dice=0.5526 | best=0.5795@26
[stage0_only_96ch] epoch 100/100 | val_dice=0.5524 | best=0.5795@26
[stage0_only_96ch] DONE | best_val_dice=0.5795 (epoch 26) | test_dice=0.5204


### Scenario 3 — MaxSig at **Stage 3 only (768ch, deepest stage)**
> Reviewer framing: *"Why stage 2 and not the deepest stage? Did you test placing the adapter at
> the very end of the backbone instead?"*


In [10]:
res, _ = run_scenario(run_name="stage3_only_768ch", maxsig_stages=(3,))
ablation_results.append(res)


/tmp/ipykernel_434032/3248516051.py:15: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage3_only_768ch] epoch 10/100 | val_dice=0.5518 | best=0.5518@10
[stage3_only_768ch] epoch 20/100 | val_dice=0.5269 | best=0.5831@18
[stage3_only_768ch] epoch 30/100 | val_dice=0.5352 | best=0.5831@18
[stage3_only_768ch] epoch 40/100 | val_dice=0.5274 | best=0.5831@18
[stage3_only_768ch] epoch 50/100 | val_dice=0.5379 | best=0.5831@18
[stage3_only_768ch] epoch 60/100 | val_dice=0.5243 | best=0.5831@18
[stage3_only_768ch] epoch 70/100 | val_dice=0.5224 | best=0.5831@18
[stage3_only_768ch] epoch 80/100 | val_dice=0.5122 | best=0.5831@18
[stage3_only_768ch] epoch 90/100 | val_dice=0.5093 | best=0.5831@18
[stage3_only_768ch] epoch 100/100 | val_dice=0.5084 | best=0.5831@18
[stage3_only_768ch] DONE | best_val_dice=0.5831 (epoch 18) | test_dice=0.5114


## GFLOPs / FPS — default configuration only\n(Stage 2 only, as used in `ConvNext-MaxSig.ipynb`)

In [11]:

gflops, fps = compute_flops_fps(default_model, input_size=(1, 1, config["img_size"], config["img_size"]))
print(f"Default config (stage=(2,)) | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


Default config (stage=(2,)) | GFLOPs: 93.133 | FPS: 45.81


In [10]:
baseline_model = ConvNeXtBinarySegmentation(
        backbone=config["backbone"], maxsig_stages=(),
    ).to(device)
gflops, fps = compute_flops_fps(baseline_model, input_size=(1, 1, config["img_size"], config["img_size"]))
print(f"baseline_model | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")

baseline_model | GFLOPs: 92.963 | FPS: 104.29


## Summary table

In [12]:
df = pd.DataFrame(ablation_results)
df = df[["run_name", "maxsig_stages", "best_val_dice", "best_epoch", "test_dice"]]
df.loc[df["run_name"] == "default_stage2_only_384ch", "GFLOPs"] = gflops
df.loc[df["run_name"] == "default_stage2_only_384ch", "FPS"] = fps
df


,run_name,maxsig_stages,best_val_dice,best_epoch,test_dice,GFLOPs,FPS
0,default_stage2_only_384ch,"(2,)",0.570663,23,0.538701,93.132556,45.813755
1,stage0_only_96ch,"(0,)",0.579544,26,0.520415,NaN,NaN
2,stage3_only_768ch,"(3,)",0.583073,18,0.511420,NaN,NaN
